In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita exibição gráfica inline no Jupyter notebook
%matplotlib inline

# Como exporto EEG gravado para interoperabilidade em NeuroAI?

Carregue uma gravação real de imaginação motora e exporte voltagens juntamente com uma tabela
de eventos com campos de linha do tempo (*timeline*), início e duração. Esta é uma verificação explícita
de fronteira MNE para PyTorch por meio do Segmenter e EegExtractor do NeuralSet, seguida por um
DataLoader do PyTorch. Instale o ``neuralset`` para executá-lo. O EEGDash lida com a descoberta;
o NeuralSet lê o arquivo de sinal adquirido e extrai as voltagens alinhadas aos eventos.
O exemplo não executa o avaliador NeuralBench. O NeuralSet preserva voltagens em float32 e a
geometria dos canais; campos ricos do MNE, como marcadores de canais ruins e histórico de filtros,
necessitam de um arquivo auxiliar separado. Máscaras de eventos não são EEG.

O subconjunto processado BNCI2014-004 nm000135 contém C3/Cz/C4 a 250 Hz e consome cerca
de 5.6 MB para o sujeito 1, sessão 0train, execução 0.


## Antes de começar

Use EEGDash, Braindecode, PyTorch e NeuralSet em um ambiente instalado compatível;
o NeuralSet é uma dependência adicional, não fornecida apenas pela interface base do EEGDash.
Suas APIs de ``Segmenter``, ``EegExtractor`` e agrupamento (*collation*) de dataset são usadas diretamente.
Esse fluxo foi testado com o NeuralSet 0.3.1, mas essa observação não substitui o respeito aos requisitos
de dependência declarados ao criar um novo ambiente.

Mantenha cerca de 6 MB de cache de sinal e permita processamento em CPU. O objetivo é a igualdade
das voltagens gravadas entre fronteiras de bibliotecas, não uma pontuação de decodificação. Nenhum modelo
ou divisão de treino/teste é necessário para verificar o adaptador. As janelas resultantes ainda
exigiriam uma divisão segura contra vazamento (*leakage-safe split*) antes de qualquer treinamento supervisionado.



In [ ]:
# Importa módulos de sistema operacional e caminhos no sistema de arquivos
import os
from pathlib import Path

# Importa bibliotecas para plotagem, computação matricial e manipulação de tabelas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa o pacote NeuralSet e DataLoader do PyTorch
import neuralset as ns
from torch.utils.data import DataLoader
# Importa gerador de janelas a partir de eventos da Braindecode
from braindecode.preprocessing import create_windows_from_events

# Importa tipos de eventos e extratores neurofisiológicos do NeuralSet
from neuralset.events.etypes import Eeg
from neuralset.extractors.neuro import MneTimedArray

# Importa classe principal de carregamento de datasets do EEGDash
from eegdash import EEGDashDataset

## Manter explícita a identidade da gravação e o tempo do evento

A consulta seleciona uma gravação genuína. ``timeline`` identifica essa gravação
ao longo de toda a tabela de eventos; usar uma linha do tempo única por arquivo impede que um evento
seja associado ao sinal de outro participante ao expandir o estudo. Os campos ``start`` e ``duration``
da tabela de eventos são expressos em segundos, enquanto MNE e Braindecode também expõem índices de amostras.

A asserção de primeira amostra igual a zero torna a convenção explícita para este exemplo.
Uma gravação cortada com origem de amostra diferente de zero requer uma conversão deliberada;
reutilizar esses deslocamentos inalterados desalinhariam sinais e eventos. A exportação em TSV
preserva as descrições BIDS originais para inspeção, mas não contém amostras de voltagem.



In [ ]:
# Configura diretório de cache persistente
cache = Path(os.environ.get("EEGDASH_CACHE_DIR", "~/.eegdash_cache")).expanduser()
# Carrega uma única gravação do sujeito 1 na sessão 0train da tarefa de imaginação motora
dataset = EEGDashDataset(
    dataset="nm000135",
    subject="1",
    session="0train",
    run="0",
    task="imagery",
    cache_dir=cache,
)
# Valida que exatamente uma gravação foi carregada
assert len(dataset.datasets) == 1
# Obtém o objeto Raw do MNE
raw = dataset.datasets[0].raw
# Exibe metadados, canais, taxa de amostragem e anotações originais
print(dataset.description.to_string(index=False))
print(raw.ch_names, raw.info["sfreq"], np.unique(raw.annotations.description))
# Assegura que o índice da primeira amostra é zero para consistência temporal
assert raw.first_samp == 0, "This example uses recording-relative event times"
# Identificador único de linha do tempo BIDS
timeline = "nm000135/sub-1/ses-0train/run-0"
# Constrói a tabela de eventos padronizada com início e duração em segundos
events = pd.DataFrame(
    {
        "timeline": timeline,
        "start": raw.annotations.onset - raw.first_time,
        "duration": raw.annotations.duration,
        "type": "Stimulus",
        "bids_description": raw.annotations.description,
    }
)
# Exporta a tabela de eventos para formato TSV
events.to_csv(cache / "plot_74_events.tsv", sep="\t", index=False)

O Braindecode extrai janelas de voltagem reais de dois segundos alinhadas a eventos.
%%
Construir uma referência de voltagem independente
-------------------------------------------------

A 250 Hz, uma janela de dois segundos possui 500 amostras. Tamanho e passo iguais
produzem janelas não sobrepostas dentro de cada intervalo elegível de imaginação; pode haver
mais de uma janela por ensaio. ``on_last_window="drop"`` evita uma janela final parcial.
Rótulos de mão esquerda e mão direita vêm das anotações gravadas, com códigos explícitos zero e um.

O layout esperado do tensor é ``(janelas, canais, amostras)`` em volts.
``MneTimedArray`` verifica uma segunda fronteira ao transportar esses valores gravados e a geometria
dos canais pela representação de array do NeuralSet. Essa conversão retém a precisão float32;
metadados ricos do MNE não são todos preservados. A asserção de ida e volta (*round-trip*)
verifica valores numéricos em vez de presumir que formatos correspondentes impliquem EEG correspondente.



In [ ]:
# Cria janelas de 2 segundos (500 amostras a 250 Hz) para mão esquerda (0) e mão direita (1)
windows = create_windows_from_events(
    dataset,
    mapping={"left_hand": 0, "right_hand": 1},
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=500,
    window_stride_samples=500,
    on_last_window="drop",
    preload=True,
)
# Obtém os metadados das janelas extraídas
metadata = windows.get_metadata()
# Empilha os dados em uma matriz numpy de voltagens (n_janelas, n_canais, 500 amostras)
volts = np.stack([windows[i][0] for i in range(len(windows))])
# Extrai os rótulos de classe (0 e 1)
labels = np.asarray([windows[i][1] for i in range(len(windows))])
# Garante que todas as voltagens são finitas e que ambas as classes estão presentes
assert np.isfinite(volts).all() and set(labels) == {0, 1}
# Converte o sinal MNE em um MneTimedArray do NeuralSet
neural_array = MneTimedArray.from_native(raw.copy().pick("eeg"))
# Reconverte para o formato nativo do MNE para validação de integridade round-trip
reconstructed = neural_array.to_native()
# Assegura precisão numérica float32 idêntica entre o dado nativo e o reconstruído
np.testing.assert_allclose(
    reconstructed.get_data(), raw.get_data(picks="eeg"), rtol=1e-6, atol=1e-12
)
# Valida preservação exata dos nomes dos canais
assert reconstructed.ch_names == raw.copy().pick("eeg").ch_names

Construir uma tabela de eventos do NeuralSet com o arquivo de EEG adquirido e âncoras de janelas.
Cada âncora e rótulo vem dos metadados reais de janelas do Braindecode.
O EegExtractor lê a carga útil do sinal; linhas de Stimulus definem apenas a temporização.
%%
Informar ao NeuralSet onde a voltagem reside
--------------------------------------------

Um evento ``Eeg`` aponta para o arquivo de sinal adquirido e sua taxa de amostragem. As
linhas de ``Stimulus`` fornecem âncoras e rótulos reais a partir dos metadados do Braindecode.
Apenas essas linhas disparam segmentos. Elas não fornecem EEG por si mesmas: o
``EegExtractor`` lê o sinal nomeado pelo evento de gravação.

O extrator usa a frequência de amostragem nativa, ordem original dos canais e
``scaler=None`` porque uma verificação de fronteira não deve introduzir uma nova transformação
de sinal. ``prepare()`` prepara os dados do extrator antes da iteração. Asserções de contagem
de segmentos e códigos de disparo verificam se a preparação não perdeu nem reordenou as janelas
rotuladas solicitadas.



In [ ]:
# Cria o evento de gravação Eeg apontando para o arquivo físico de dados brutos
recording_event = Eeg(
    filepath=str(raw.filenames[0]),
    start=0.0,
    duration=raw.n_times / raw.info["sfreq"],
    frequency=raw.info["sfreq"],
    subject="1",
    timeline=timeline,
).to_dict()
# Constrói a tabela de âncoras de estímulo baseada nas janelas geradas pelo Braindecode
anchors = pd.DataFrame(
    {
        "type": "Stimulus",
        "timeline": timeline,
        "start": metadata["i_start_in_trial"].to_numpy() / raw.info["sfreq"],
        "duration": 2.0,
        "code": labels,
    }
)
# Padroniza os eventos unificados no esquema do NeuralSet
neural_events = ns.events.standardize_events(
    pd.concat([pd.DataFrame([recording_event]), anchors], ignore_index=True)
)
# Configura o Segmenter para extrair janelas de 2.0s sem reescalonamento (scaler=None)
segmenter = ns.Segmenter(
    trigger_query="type == 'Stimulus'",
    start=0.0,
    duration=2.0,
    extractors={
        "eeg": ns.extractors.EegExtractor(
            frequency="native",
            scaler=None,
            channel_order="original",
            mne_cpus=1,
        )
    },
)
# Aplica a segmentação aos eventos padronizados
neural_dataset = segmenter.apply(neural_events)
# Prepara o dataset para iteração
neural_dataset.prepare()
# Valida que a quantidade de segmentos coincide exatamente com a contagem de janelas do Braindecode
assert len(neural_dataset) == len(windows)
# Valida que todos os códigos de trigger coincidem com os rótulos de classe originais
np.testing.assert_array_equal(
    [segment.trigger.code for segment in neural_dataset.segments], labels
)

## Agrupar em lotes com o contrato de agrupamento (*collation*) do dataset

Itens do NeuralSet são exemplos estruturados em vez de simples arrays NumPy,
portanto o DataLoader usa a própria função ``collate_fn`` do dataset. Lotes de 16 delimitam
a memória; zero workers mantêm esta pequena demonstração direta e ``shuffle=False`` preserva
a ordem de referência. Voltagens são lidas de ``batch.data["eeg"]``, e não de um campo de máscara de evento.

A comparação do array completo testa todas as janelas contra dados extraídos de forma independente
pelo Braindecode. Uma fatia direta de amostras do MNE adiciona uma verificação temporal explícita da
primeira janela. A tolerância relativa de ``1e-6`` e tolerância absoluta de ``1e-12`` volts acomodam a
conversão float32 sem ocultar erros de milivolts/microvolts, canais reordenados ou deslocamentos de amostras.



In [ ]:
# Cria o DataLoader do PyTorch usando a função collate_fn nativa do dataset do NeuralSet
loader = DataLoader(
    neural_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=neural_dataset.collate_fn,
)
# Concatena todas as voltagens recuperadas pelo DataLoader
restored = np.concatenate([batch.data["eeg"].numpy() for batch in loader])
# Valida que todos os valores coincidem estritamente com os extraídos pelo Braindecode
np.testing.assert_allclose(restored, volts, rtol=1e-6, atol=1e-12)
# Compara a primeira janela diretamente com uma fatia de amostras do MNE de forma independente
start = int(metadata.iloc[0]["i_start_in_trial"])
expected = raw.get_data(picks="eeg", start=start, stop=start + 500)
np.testing.assert_allclose(restored[0], expected, rtol=1e-6, atol=1e-12)
# Exibe as dimensões finais e a distribuição das classes observadas
print("NeuralSet voltage tensor (windows, channels, samples):", restored.shape)
print("Observed class counts:", np.unique(labels, return_counts=True))
# Salva os tensores e metadados estruturados em arquivo compactado .npz
np.savez(
    cache / "plot_74_voltages.npz",
    volts=restored,
    labels=labels,
    sfreq=raw.info["sfreq"],
    channels=raw.ch_names,
    timeline=timeline,
)

In [ ]:
# Cria figura com dois subgráficos comparando as voltagens no canal C3
fig, axes = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
# Eixo temporal em segundos para as 500 amostras (2s)
times = np.arange(500) / raw.info["sfreq"]
# Subgráfico superior: sobreposição da curva MNE e da curva NeuralSet -> DataLoader
axes[0].plot(times, expected[0] * 1e6, label="MNE voltage")
axes[0].plot(times, restored[0, 0] * 1e6, "--", label="NeuralSet → DataLoader")
axes[0].set(ylabel="C3 (µV)")
axes[0].legend()
# Subgráfico inferior: diferença ponto a ponto (deve ser essencialmente zero dentro de float32)
axes[1].plot(times, (restored[0, 0] - expected[0]) * 1e6)
axes[1].set(xlabel="Time (s)", ylabel="Difference (µV)")
# Exibe o gráfico
plt.show()

## Reutilizar a exportação sem perder seu significado

O arquivo NPZ armazena voltagens, rótulos, taxa de amostragem, nomes dos canais e a linha do tempo;
o TSV preserva a temporização dos eventos e descrições originais. O gráfico compara valores do
primeiro canal extraídos de forma independente e sua diferença real. Trata-se de uma verificação numérica
de interoperabilidade nesta gravação, não de uma validação de todo formato de arquivo ou de toda
configuração de pré-processamento.

Para estender a múltiplas gravações, construa um evento ``Eeg`` e uma linha do tempo distintos para
cada arquivo e mantenha identidades de sujeito e ensaio original ao lado das janelas. Divida sujeitos
ou ensaios completos antes do treinamento, pois múltiplas janelas de um mesmo ensaio compartilham a
mesma fonte. Decida explicitamente como preservar marcadores de canais ruins, anotações e histórico
de pré-processamento antes de tratar a exportação como uma substituição do objeto MNE original.

Exemplo relacionado de fronteira de dados: [Braindecode training on MNE epochs](https://braindecode.org/stable/auto_examples/model_building/plot_basic_training_epochs.html).



## Continuar para o pré-treinamento autosupervisionado

Siga o guia do NeuroAI [Training a model: masked prediction on EEG](https://facebookresearch.github.io/neuroai/neuralbench/auto_examples/biosignal_challenge_2026/plot_pretrain_mae.html)
para um loop de pré-treino detalhado, exportação de checkpoint e avaliação posterior.
A predição mascarada reconstrói porções ocultas de EEG gravado; os rótulos de imaginação acima
não são alvos de reconstrução.

1. Siga as instruções de instalação do guia para o projeto ``ssl_example`` do repositório
   e suas dependências de treinamento. A partir de seu diretório ``neuraltrain-repo``, comece com
   a execução de depuração em dados reais:

```console
python -m ssl_example.grids.test_run
```
2. Configure os estudos, divisões de sujeitos e caminhos de dados/cache/saída antes de usar os
   comandos de download e treinamento completo do guia. Seu corpus padrão necessita de cerca de 1.1 TB;
   a execução de depuração usa uma gravação pequena real do MNE.

3. Use o caminho impresso de ``encoder.ckpt`` no comando de avaliação posterior do guia. Ajuste
   a configuração do codificador e o pré-processamento ao pré-treino; uma perda de reconstrução isolada
   não estabelece o desempenho de decodificação.

Adaptar esta conversão exige mais do que passar ``loader`` para aquele script.
Aqui os lotes contêm ``eeg`` a 250 Hz, com 500 amostras e âncoras de estímulo.
O guia constrói lotes de ``input``, posições de canais e janelas com passo pela gravação a 120 Hz
para seu codificador de patches. Adapte a configuração de estudo/extrator às gravações adquiridas,
mantenha a separação no nível do sujeito e verifique o contrato resultante dos lotes antes de treinar.
O NPZ é uma exportação de voltagens, não um checkpoint pré-treinado ou um estudo registrado no NeuroAI.

